# cv-data-lesson 自動GPU優先・CPUフォールバック受け入れテスト

授業用Notebookとは別の確認用Notebookです。CPUランタイムとT4 GPUランタイムで、このNotebookをそれぞれ最初から実行します。`device="auto"`がGPUを優先し、GPUがない場合に警告を出してCPUへ切り替わること、ResNet・DETR・SAM・画像生成が実モデルで動くことを確認します。

OCRはPaddlePaddleとの依存関係を分けるため、`ocr_acceptance_test.ipynb`で確認します。授業用Notebookは変更しません。

In [ ]:
LIBRARY_REF = "f5034d2845401198fc4b5de88b7dbf12afdc7d47"
%pip install -q "cv-data-lesson[torch,vision,diffusion] @ git+https://github.com/708san/CV_data.git@f5034d2845401198fc4b5de88b7dbf12afdc7d47"

In [ ]:
import gc
import importlib.metadata as metadata
import warnings
from pathlib import Path

import torch
from IPython.display import display
from PIL import Image, ImageDraw

from cvlesson import generation, models, vision

HAS_GPU = bool(torch.cuda.is_available())
print("version:", metadata.version("cv-data-lesson"))
print("runtime:", "GPU" if HAS_GPU else "CPU")
print("ResNet devices:", models.available_devices())
print("Vision devices:", vision.available_devices())
print("Generation devices:", generation.available_devices())
if HAS_GPU:
    assert models.available_devices()["cuda:0"]
    assert vision.available_devices()["cuda:0"]
else:
    assert models.available_devices()["cpu"]
    assert vision.available_devices()["cpu"]
MODEL_CACHE = Path("/content/cvlesson-auto-model-cache")
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

## ResNet分類（`auto`）

In [ ]:
images = []
labels = []
for index, color in enumerate(("red", "red", "blue", "blue")):
    image = Image.new("RGB", (96, 96), color)
    images.append(image)
    labels.append("赤" if color == "red" else "青")

classifier = models.ResNetClassifier(
    device="auto", epochs=1, batch_size=2, weights=None, verbose=True
)
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter("always")
    classifier.fit(images, labels)
if not HAS_GPU:
    assert any("CPU" in str(item.message) for item in captured)
print("ResNet device:", classifier.device)
print("ResNet predictions:", classifier.predict(images))
assert len(classifier.predict(images)) == 4

## DETR物体検出とSAM領域分割（`auto`）

CPUでは同じ固定モデルをCPUで実行するため、T4より時間がかかります。

In [ ]:
scene = Image.new("RGB", (256, 256), "white")
draw = ImageDraw.Draw(scene)
draw.rectangle((40, 50, 210, 190), fill="red")
scene_path = Path("/content/cvlesson-auto-scene.png")
scene.save(scene_path)

detector = vision.load_detector(device="auto", cache_dir=MODEL_CACHE)
segmenter = vision.load_segmenter(device="auto", cache_dir=MODEL_CACHE)
print("DETR device:", detector.device)
print("SAM device:", segmenter.device)
assert detector.device == ("cuda:0" if HAS_GPU else "cpu")
assert segmenter.device == ("cuda:0" if HAS_GPU else "cpu")
detections = detector.predict(scene_path, threshold=0.1)
print("detections:", len(detections))
mask_result = segmenter.predict(scene_path, box=(40, 50, 210, 190))
print("SAM mask:", mask_result.mask.shape, mask_result.mask.dtype)
assert mask_result.mask.shape == (256, 256)

## 画像生成（`auto`）

GPUではSDXL、CPUではTiny-SDを使います。CPUではLoRAを使わず、text-to-imageだけを確認します。Tiny-SDでLoRAを指定した場合とimage-to-imageを指定した場合は、黙って無視せずエラーになることも確認します。

In [ ]:
generator = generation.load(
    model="auto", device="auto", lora=None, cache_dir=MODEL_CACHE
)
print("generation model/device:", generator.model, generator.device)
assert generator.model == ("sdxl" if HAS_GPU else "tiny-sd")
assert generator.device == ("cuda:0" if HAS_GPU else "cpu")
image = generator.generate(
    "simple red apple on a white table, educational illustration",
    seed=42, num_inference_steps=2, guidance_scale=4.0, width=512, height=512
)
display(image)
assert image.size == (512, 512)
if not HAS_GPU:
    try:
        generation.load(model="tiny-sd", device="cpu", lora="fluent_emoji_3d")
    except ValueError as exc:
        print("Tiny-SD LoRA拒否:", exc)
    else:
        raise AssertionError("Tiny-SDがLoRAを受け付けました")
    del generator
    gc.collect()

In [ ]:
print("auto device model acceptance: OK")
print("OCRは別ランタイムの ocr_acceptance_test.ipynb で確認します。")